In [ ]:
import sys

sys.path.append("../src")

import altair as alt
alt.renderers.enable("jupyter", offline=True)
alt.data_transformers.disable_max_rows()

import gc
import os
import polars as pl
import polars.selectors as cs
import warnings
from sklearn.model_selection import GroupKFold
from dataclasses import asdict

import lightgbm as lgb
from dotenv import load_dotenv

import wandb
from pathlib import Path

from config import cfg
from data.data_class import Dfs
from data.data_process import add_fold, get_Xy
from data.simple_feature_eng import preprocess
from models.lgb import train_model

# warnings.filterwarnings("ignore")
# warnings.simplefilter("ignore")

cfg.train_path = Path("../data/train.csv")
cfg.test_path = Path("../data/test.csv")
cfg.pltpd_path = Path("../data/podcast_dataset.csv")


df_test = pl.read_csv(cfg.test_path)

df_train = pl.read_csv(cfg.train_path)
df_train = df_train.filter(pl.col("Number_of_Ads").is_not_null())

# df_train = df_train.drop("id")
df_train = add_fold(df_train)
df_train = preprocess(df_train)

df_pltpd = pl.read_csv(cfg.pltpd_path)
df_pltpd = df_pltpd.filter(pl.col("Episode_Length_minutes").is_not_null())
df_pltpd = df_pltpd.with_columns(pl.col("Number_of_Ads").cast(pl.Float64))
df_pltpd = add_fold(df_pltpd)
df_pltpd = preprocess(df_pltpd)
df_pltpd = df_pltpd.with_columns(pl.Series(range(1_000_000, 1_000_000 + len(df_pltpd))).alias("id"))
# df_pltpd

df = df_train.clone()
df

In [ ]:
import numpy as np

def calculate_rmse(actual, predicted):
    squared_diff = (actual - predicted) ** 2
    mean_squared_diff = squared_diff.mean()
    rmse = np.sqrt(mean_squared_diff)
    return rmse

def optimize_scaling_factor(input_data, target_data, n_searches=20):
    initial_x = target_data.mean() / input_data.mean()
    
    # Define search range
    lower_bound = initial_x * 0.5
    upper_bound = initial_x * 1.5
    
    # Perform n binary searches
    for _ in range(n_searches):
        mid_point = (lower_bound + upper_bound) / 2
        
        delta = (upper_bound - lower_bound) * 0.1
        
        lower_x = mid_point - delta
        upper_x = mid_point + delta
        
        lower_rmse = calculate_rmse(target_data, input_data * lower_x)
        upper_rmse = calculate_rmse(target_data, input_data * upper_x)
        
        if lower_rmse < upper_rmse:
            upper_bound = mid_point
        else:
            lower_bound = mid_point
    
    best_x = (lower_bound + upper_bound) / 2
    best_rmse = calculate_rmse(target_data, input_data * best_x)
    
    return best_x, best_rmse

import time
for i in range(0, 100, 1):
    start_time = time.time()
    x_optimal = optimize_scaling_factor(df_train["Episode_Length_minutes"], df_train["Listening_Time_minutes"], n_searches=i)
    print(i, "\t", x_optimal, "\t",  time.time() - start_time)